In [2]:
import re
import json
from glob import glob
import pandas as pd

In [90]:
path_to_folder = "../CFDE/"
schema_name = "cfde"

In [91]:
node_schema = []
node_pattern = "(?P<directory>.+)/(?P<label>.+)\.(?P<entity>.+)\.csv"
for filename in glob(f"{path_to_folder}/*.nodes.csv"):
    match = re.match(node_pattern, filename).groupdict()
    label = match["label"].replace("_", " ")
    df = pd.read_csv(filename)
    display = []
    for i in df.columns:
        if i == "id":
            display.append({
                'label': '%s ID'%label,
                'text': "${%s}"%i,
                'type': 'text'
            })
        else:
            val = str(df.at[0, i])
            if val.startswith("http"):
                display.append({
                    'label': i,
                    'text': "${%s}"%i,
                    'href': "${%s}"%i,
                    'type': 'link'
                })
            else:
                display.append({
                    'label': i,
                    'text': "${%s}"%i,
                    'type': 'text'
                })
    examples = list(df.head(2).label)
    node = {
        "node": label,
        "example": examples,
        "display": display,
        "relation": [],
        "search": [
            "id",
            "label",
        ],
        # "color": "#c5e1a5"
    }
    node_schema.append(node)
        

<>:2: SyntaxWarning: invalid escape sequence '\.'
<>:2: SyntaxWarning: invalid escape sequence '\.'
/var/folders/4l/73ssnfdj1mg_zmt5ghdmqjjh0000gp/T/ipykernel_60390/2820105713.py:2: SyntaxWarning: invalid escape sequence '\.'
  node_pattern = "(?P<directory>.+)/(?P<label>.+)\.(?P<entity>.+)\.csv"


In [92]:
edge_schema = []
edge_pattern = "(?P<directory>.+)/(?P<source_type>.+)\.(?P<relation>.+)\.(?P<target_type>.+)\.(?P<entity>.+)\.csv"

for filename in glob(f"{path_to_folder}/*.edges.csv"):
   match = re.match(edge_pattern, filename).groupdict()
   entity = match["entity"]
   source_type = match["source_type"]
   relation = match["relation"].replace("_", " ")
   edge = {
      "match": [relation],
      "selected": True,
   }
   df = pd.read_csv(filename)
   display = []
   for i in df.columns:
      if i == "id":
         display.append({
               'label': '%s ID'%label,
               'text': "${%s}"%i,
               'type': 'text'
         })
      else:
         val = str(df.at[0, i])
         if val.startswith("http"):
               display.append({
                  'label': i.replace("_", " "),
                  'text': "${%s}"%i,
                  'href': "${%s}"%i,
                  'type': 'link'
               })
         else:
               display.append({
                  'label': i.replace("_", " "),
                  'text': "${%s}"%i,
                  'type': 'text'
               })
         edge["display"] = display
   edge_schema.append(edge)
         

<>:2: SyntaxWarning: invalid escape sequence '\.'
<>:2: SyntaxWarning: invalid escape sequence '\.'
/var/folders/4l/73ssnfdj1mg_zmt5ghdmqjjh0000gp/T/ipykernel_60390/4003153242.py:2: SyntaxWarning: invalid escape sequence '\.'
  edge_pattern = "(?P<directory>.+)/(?P<source_type>.+)\.(?P<relation>.+)\.(?P<target_type>.+)\.(?P<entity>.+)\.csv"


In [93]:
len(edge_schema)

5

## Additional schema

The next part are miscellaneous schemas that you can use to modify our UI. Notice the tabs field, this is where we define the pages in our knowledge graph as well as the components that you want to render.

Currently we have the following components available:
1. KnowledgeGraph: this renders the single and two term search components
2. Markdown: this is for rendering static markdown files, useful for tutorials
3. Enrichment: For enrichment page.
   
Each component takes specific props like init_function that initializes/prefetch the static props for the page. For this example, we only look at the KnowledgeGraph component. For more information on how to use the other components you can check: https://maayanlab.cloud/enrichr-kg/api/knowledge_graph/schema

In [94]:
# First you need to modify this based on your dataset. 
# This will be the initial query that the users will see when visiting your site
initial_query = {
	"start": "authors",
	"start_term": "Avi Ma'ayan",
	"start_field": "label"
}

In [95]:
title = "Demo-KG"
icon = "https://minio.dev.maayanlab.cloud/enrichr-kg/enrichr-kg.png"
rest = {
    "header": {
		"header": title,
		"title": title,
		"background": {
			"backgroundColor": "#000",
			"contrastText": "#FFF"
		},
		"icon": {
			"src": icon,
			"favicon": icon,
			"faviconTitle": title,
			"alt": title,
			"key": "%s_logo"%title,
			"width": 100,
			"height": 60
		},
		"tabs": [
			{
				"endpoint": "/",
				"label": "Term & Gene Search",
				"type": "page",
				"component": "KnowledgeGraph",
				"props": {
					"init_function": "initialize_kg",
					"initial_query": initial_query,
					"subheader": {
						"url_field": "filter",
						"query_field": "relation"
					}
				}
				}
		]
	},
	"footer": {
		"style": {
            "background": "#000",
            "color": "#FFF"
        },
		"layout": [
			[
				{
                    "component": "logo",
                    "props": {
                        "src": "https://minio.dev.maayanlab.cloud/enrichr-kg/enrichr-kg.png",
                        "alt": "enrichr-kg",
                        "title": "Enrichr-KG",
                        "color": "inherit",
                        "size": "small"
                    }
                },
				{
                    "component": "github",
                    "props": {
                        "code": "https://github.com/MaayanLab/enrichr-kg",
                        "issues": "https://github.com/MaayanLab/enrichr-kg-issues/issues/new"
                    }
                }
			],
			[
				{
					"component": "icon",
					"props": {
						"type": "icon",
						"key": "lab_icon",
						"src": "https://minio.dev.maayanlab.cloud/enrichr-kg/maayanlab.png",
						"alt": "MaayanLab",
						"href": "https://labs.icahn.mssm.edu/maayanlab/",
						"height": 60,
						"width": 150
					}
				  }
			],
			[
				{
					"component": "icon",
					"props": {
						"type": "icon",
						"key": "center_icon",
						"src": "https://minio.dev.maayanlab.cloud/enrichr-kg/bioinformatics.png",
						"alt": "COB",
						"href": "https://icahn.mssm.edu/research/bioinformatics",
						"height": 60,
						"width": 150
					}
				  }
			]
		],
		"footer_text": "<span style='color:white'>Please acknowledge Enrichr-KG in your publications by citing the following reference:<br><a style='color: cyan' href='https://doi.org/10.1093/nar/gkad393'>Evangelista JE, Xie Z, Marino GB, Nguyen N, Clarke DJB, Ma'ayan A. Enrichr-KG: bridging enrichment analysis across multiple libraries. Nucleic Acids Res. 2023 May 11:gkad393. doi: 10.1093/nar/gkad393. PMID: 37166973.</a></span>"
	},
	"ui_theme": "enrichr_kg_theme"
}

## Putting it all together

In [96]:
schema = {
    "nodes": node_schema,
    "edges": edge_schema,
    **rest
}

with open(f"./schema_{schema_name}.json", 'w') as o:
    o.write(json.dumps(schema, indent=4))